# TP – Pipeline Machine Learning  
## M1 IA & Big Data  

**Étudiant :** Alireza KARBALAY MOHAMMADI  
**Enseignante :** Wejdeh ABDALLAH  

---

##  Objectif

L’objectif de ce projet est de construire un pipeline complet de machine learning afin de prédire la note finale des élèves (G3) à partir de leurs caractéristiques scolaires et socio-démographiques.

---

##  Méthodologie

Dans ce projet, nous allons :

1. Charger et explorer le dataset  
2. Effectuer les prétraitements (encodage des variables catégorielles et normalisation des variables numériques)  
3. Construire et comparer trois modèles :
   - Régression linéaire  
   - Arbre de décision  
   - Réseau de neurones (MLPRegressor)  
4. Entraîner les modèles et les évaluer avec les métriques suivantes :
   - MAE (Mean Absolute Error)  
   - RMSE (Root Mean Squared Error)  
   - R² (coefficient de détermination)  
5. Interpréter les résultats et déterminer le modèle le plus adapté  

---

##  Dataset

Nous utilisons le dataset **Student Performance** provenant du UCI Machine Learning Repository.  
La variable cible est **G3 (note finale)**.

---

##  Résultat attendu

À la fin de ce projet, nous identifierons le modèle le plus performant et discuterons de ses avantages, de ses limites et de son applicabilité.

##  Importation des bibliothèques

Dans cette section, nous importons toutes les bibliothèques nécessaires à la réalisation du projet.

- **pandas** : pour la manipulation des données  
- **scikit-learn** : pour le prétraitement, la construction des modèles et l’évaluation  
- **numpy** : pour les calculs numériques  

Ces outils nous permettront de construire un pipeline complet de machine learning, depuis la préparation des données jusqu’à l’évaluation des modèles.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

##  Exploration des données

Dans cette étape, nous explorons le dataset afin de mieux comprendre sa structure et son contenu.

- **head()** : permet d’afficher les premières lignes du dataset  
- **shape** : donne le nombre de lignes et de colonnes  
- **info()** : fournit des informations sur les types de variables et les valeurs manquantes  
- **describe()** : donne des statistiques descriptives des variables numériques  
- **columns** : affiche les noms des variables  

Cette exploration est essentielle pour identifier les types de données (numériques ou catégorielles) et préparer les étapes de prétraitement.

In [ ]:
df=pd.read_csv("student-mat.csv", sep=";")
df.head(5)

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [ ]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')

In [ ]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

In [ ]:
df.shape

(395, 33)

In [ ]:
df.describe()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,16.696203,2.749367,2.521519,1.448101,2.035443,0.334177,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861,10.908861,10.713924,10.415190
std,1.276043,1.094735,1.088201,0.697505,0.839240,0.743651,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096,3.319195,3.761505,4.581443
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.000000,0.000000,0.000000
25%,16.000000,2.000000,2.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,8.000000,9.000000,8.000000
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,11.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000,19.000000,19.000000,20.000000


###  Nettoyage des données

Après exploration du dataset, nous constatons que les données sont propres, sans valeurs manquantes et déjà bien structurées.

Par conséquent, aucune étape de nettoyage supplémentaire n’a été nécessaire dans ce projet.

##  Identification des variables

Dans cette étape, nous séparons les variables explicatives (X) et la variable cible (y).

- **G3** est définie comme la variable cible (note finale).  
- Les autres variables constituent les variables explicatives.

Ensuite, nous distinguons :
- Les variables **numériques** (ex : âge, absences, notes précédentes…)  
- Les variables **catégorielles** (ex : sexe, école, adresse…)

Cette séparation est essentielle pour appliquer les bons prétraitements dans le pipeline (normalisation pour les variables numériques et encodage pour les variables catégorielles).

In [ ]:

target = "G3"

X = df.drop(columns=[target])
y = df[target]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns

cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical columns:")
print(num_cols)

print("\nCategorical columns:")
print(cat_cols)

Numerical columns:
Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2'],
      dtype='object')

Categorical columns:
Index(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic'],
      dtype='object')


### Prétraitement des variables

Les variables doivent être transformées avant d’être utilisées par les modèles de machine learning.

#### Variables catégorielles
Les variables catégorielles ne peuvent pas être utilisées directement.  
Nous utilisons **OneHotEncoder** pour les transformer en variables numériques (vecteurs binaires).  

L’option `handle_unknown="ignore"` permet de gérer les nouvelles catégories dans les données de test sans provoquer d’erreur.

#### Variables numériques
Les variables numériques sont normalisées à l’aide de **StandardScaler**.  

Cette transformation permet de mettre toutes les variables à la même échelle, ce qui améliore les performances de certains modèles comme la régression linéaire et les réseaux de neurones.

In [ ]:
cat_transformer = OneHotEncoder(handle_unknown="ignore")

num_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols)
    ]
)

###  Pipeline de régression linéaire

Nous construisons un pipeline combinant les étapes de prétraitement et le modèle de régression linéaire.

Le préprocesseur applique automatiquement :
- l’encodage des variables catégorielles (OneHotEncoder)  
- la normalisation des variables numériques (StandardScaler)  

Ensuite, le modèle **LinearRegression** est entraîné sur les données transformées.

Ce pipeline permet d’automatiser toutes les étapes et d’assurer une meilleure cohérence dans le traitement des données.

In [ ]:
linear_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

### Division des données (Train/Test)

Le dataset est divisé en deux parties :

- **80 % pour l’entraînement (train)**  
- **20 % pour le test (test)**  

Cette séparation permet d’entraîner le modèle sur une partie des données et d’évaluer ses performances sur des données jamais vues.

Le paramètre `random_state=42` garantit la reproductibilité des résultats.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Entraînement du modèle

Le pipeline est entraîné sur les données d’entraînement à l’aide de la méthode `fit()`.

Cette étape permet :
- d’apprendre les transformations (encodage et normalisation)  
- d’ajuster le modèle de régression linéaire aux données  

Toutes les étapes du pipeline sont exécutées automatiquement.

In [ ]:
linear_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic'],
      dtype='object'))])),
                ('model', LinearRegression())])

### Prédiction

Après l’entraînement, le modèle est utilisé pour prédire la variable cible sur les données de test à l’aide de la méthode `predict()`.

Ces prédictions seront ensuite utilisées pour évaluer les performances du modèle.

In [ ]:
y_pred = linear_pipeline.predict(X_test)

###  Évaluation du modèle

Les performances du modèle sont évaluées à l’aide de trois métriques :

- **MAE (Mean Absolute Error)** : mesure l’erreur moyenne entre les valeurs réelles et les prédictions  
- **RMSE (Root Mean Squared Error)** : pénalise davantage les grandes erreurs  
- **R² (coefficient de détermination)** : mesure la qualité de l’ajustement du modèle  

Ces métriques permettent de comparer les différents modèles et de déterminer le plus performant.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 1.64666561971475
RMSE: 2.3783697847961367
R2: 0.7241341236974022


###  Pipeline de l’arbre de décision

Nous construisons un pipeline pour le modèle **DecisionTreeRegressor**.

Contrairement à la régression linéaire, les arbres de décision ne nécessitent pas de normalisation des variables numériques.  
Ainsi :

- Les variables catégorielles sont encodées avec **OneHotEncoder**  
- Les variables numériques sont conservées telles quelles (`remainder="passthrough"`)  

Le modèle **DecisionTreeRegressor** est ensuite entraîné sur ces données.

Ce pipeline permet de traiter correctement les différents types de variables tout en respectant les spécificités de l’arbre de décision.

In [ ]:
tree_pipeline = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ],
        remainder="passthrough"
    )),
    ("model", DecisionTreeRegressor(random_state=42))
])

###  Entraînement et prédiction (Arbre de décision)

Le pipeline de l’arbre de décision est entraîné sur les données d’entraînement à l’aide de la méthode `fit()`.

Ensuite, le modèle est utilisé pour effectuer des prédictions sur les données de test avec la méthode `predict()`.

Ces prédictions permettront d’évaluer les performances du modèle et de les comparer avec les autres approches.

In [ ]:
tree_pipeline.fit(X_train, y_train)

y_pred_tree = tree_pipeline.predict(X_test)

###  Évaluation du modèle (Arbre de décision)

Les performances du modèle d’arbre de décision sont évaluées à l’aide des mêmes métriques :

- **MAE (Mean Absolute Error)**  
- **RMSE (Root Mean Squared Error)**  
- **R² (coefficient de détermination)**  

Cela permet de comparer directement ce modèle avec les autres (régression linéaire et MLP) et d’analyser ses performances ainsi que son éventuel surapprentissage.

In [ ]:
mae_tree = mean_absolute_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mean_squared_error(y_test, y_pred_tree))
r2_tree = r2_score(y_test, y_pred_tree)

print("MAE:", mae_tree)
print("RMSE:", rmse_tree)
print("R2:", r2_tree)

MAE: 1.2784810126582278
RMSE: 2.3411562652304627
R2: 0.7326993404807303


###  Pipeline du réseau de neurones (MLP)

Nous construisons un pipeline pour le modèle **MLPRegressor** (réseau de neurones).

Le préprocesseur applique :
- l’encodage des variables catégorielles (OneHotEncoder)  
- la normalisation des variables numériques (StandardScaler)  

Ces transformations sont nécessaires pour les réseaux de neurones afin d’assurer une meilleure convergence.

Le modèle est configuré avec :
- deux couches cachées `(100, 50)` pour capturer des relations complexes  
- `max_iter=500` pour permettre un apprentissage suffisant  
- `random_state=42` pour garantir la reproductibilité  

Ce pipeline permet de modéliser des relations non linéaires plus complexes que les autres approches.

In [ ]:
mlp_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", MLPRegressor(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=42
    ))
])

### Entraînement et prédiction (MLP)

Le pipeline du réseau de neurones est entraîné sur les données d’entraînement à l’aide de la méthode `fit()`.

Ensuite, le modèle est utilisé pour générer des prédictions sur les données de test avec la méthode `predict()`.

Ces prédictions seront utilisées pour évaluer les performances du modèle et les comparer avec celles des autres modèles.

In [ ]:
mlp_pipeline.fit(X_train, y_train)

y_pred_mlp = mlp_pipeline.predict(X_test)

###  Évaluation du modèle (MLP)

Les performances du modèle MLP sont évaluées à l’aide des mêmes métriques :

- **MAE (Mean Absolute Error)**  
- **RMSE (Root Mean Squared Error)**  
- **R² (coefficient de détermination)**  

Cela permet de comparer ce modèle avec la régression linéaire et l’arbre de décision.

Le MLP est capable de capturer des relations plus complexes, mais ses performances dépendent fortement des paramètres choisis et peuvent varier.

In [ ]:
mae_mlp = mean_absolute_error(y_test, y_pred_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
r2_mlp = r2_score(y_test, y_pred_mlp)

print("MAE:", mae_mlp)
print("RMSE:", rmse_mlp)
print("R2:", r2_mlp)

MAE: 1.645419231712613
RMSE: 2.60011925849287
R2: 0.6702948269271092


###  Tableau comparatif des modèles

Nous regroupons les résultats des trois modèles dans un tableau récapitulatif.

Ce tableau permet de comparer directement les performances de chaque modèle selon les métriques :
- MAE  
- RMSE  
- R²  

Il facilite l’identification du modèle le plus performant et sert de base pour l’analyse et la conclusion.

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "MLP"],
    "MAE": [mae, mae_tree, mae_mlp],
    "RMSE": [rmse, rmse_tree, rmse_mlp],
    "R2": [r2, r2_tree, r2_mlp]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,1.646666,2.378370,0.724134
1,Decision Tree,1.278481,2.341156,0.732699
2,MLP,1.645419,2.600119,0.670295


##  Interprétation des résultats

Après comparaison des trois modèles, nous pouvons observer des différences importantes en termes de performance.

Le modèle de régression linéaire donne des résultats corrects mais reste limité, car il suppose une relation linéaire entre les variables. Il peut donc manquer certaines relations plus complexes dans les données.

L’arbre de décision présente généralement de très bonnes performances, avec un R² élevé et des erreurs faibles. Cependant, ce modèle a tendance à surapprendre (overfitting), ce qui signifie qu’il peut être très performant sur les données de test mais moins robuste sur de nouvelles données.

Le modèle MLP (réseau de neurones) permet de capturer des relations non linéaires plus complexes. Ses performances sont souvent équilibrées entre celles de la régression linéaire et de l’arbre de décision. Toutefois, il est plus difficile à interpréter et dépend fortement des paramètres choisis.

Il est également important de noter que les variables G1 et G2 (notes précédentes) ont une forte influence sur la prédiction de G3, ce qui facilite le travail des modèles.

En conclusion, le choix du modèle dépend de l’objectif :
- Si l’on recherche la simplicité et l’interprétabilité → régression linéaire  
- Si l’on privilégie la performance brute → arbre de décision (avec risque de surapprentissage)  
- Si l’on souhaite un compromis entre complexité et généralisation → MLP